# PromptOptimization

**Module 13 · Lesson 13.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why Bloat Costs Twice
- Find the Waste: Analyze by Component
- Trim the Input
- Cut the Output
- Automated Compression: LLMLingua
- Structure for the Cache

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

## Eight million wasted tokens a day, before anyone types

> 💡 **Info**
>
> Your assistant works beautifully, so nobody looks at the prompt. But that prompt carries an **eight-hundred-token system prompt on every single request** — a wall of instructions, examples, and polite filler that never changes. At ten thousand requests a day, that is **eight million tokens a day** you pay for before the user has typed a single word, and most of it is bloat: redundant instructions, “please kindly ensure that you…”, three examples where one would do, and context the model already knows. Lesson 13.1 gave you the uncomfortable law — cost is tokens times price — and the highest-leverage response is not a cheaper model or a discount; it is a **leaner prompt**, because the prompt runs on *every* call, forever. This lesson is the craft of cutting tokens without cutting quality: find where the tokens hide, trim the input with concise instructions and fewer examples, cut the output (the bigger lever, since output is priced several times higher), hand the un-trimmable contexts to an automatic compressor, structure the prompt so the expensive repeated part is cached, and — the step everyone skips — measure every cut against an eval set so you keep the win only when quality holds.

### 🎯 What you will be able to do after this lesson

- **Build** prompt-optimization tooling — a component analyzer, an input trimmer, an output-format optimizer, a token-informativeness compressor, a cache-aligned structurer, and an A/B quality guard — as runnable models plus an illustrative pipeline.

- **Compare** a verbose prompt with a concise one, and free-form output with structured output, on tokens and cost.

- **Operate** few-shot pruning, an automated compressor with a budget, a stable-prefix-first layout, and a token-budget guardrail.

- **Evaluate** a prompt optimization: did it cut tokens meaningfully AND hold quality on the eval set?

> 📦 **Info**
>
> ✅ Before you startIn **13.1** you learned cost is tokens times price, output costs several times more than input, and the retrieved context is most of what you pay for — this lesson trims exactly those. In **12.4** caching charged a fraction of the price on an identical leading prefix, which is why prompt *structure* matters here. The **eval set** you A/B a prompt against is built in **Lesson 14.4** and watched in **12.8**. Prompt optimization for *quality* (DSPy) is **Lesson 5.6**; this lesson is optimization for *cost*.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🧸 **Analogy**
>
> Prompt optimization is **packing a suitcase you check on every single flight**. You do not pay once; you pay the overweight fee every time you fly (the system prompt on every request). So you lay everything out and weigh it (analyze by component), you cut the three sweaters down to one and swap the bulky toiletries for travel sizes (trim the input, prune the few-shots), you stop packing the hotel towels you will be given anyway (drop what the model knows), and you put the things you always bring in a fixed carry-on you never repack (the stable, cached prefix). **Where it breaks down:** a suitcase is packed once, while your prompt is re-weighed on every call — which is exactly why a small trim pays out thousands of times.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A longer, more detailed prompt is a better prompt.”** Verbose prompts add cost and ambiguity; a concise, single-purpose instruction is usually cheaper *and* clearer.
> - **“Only the input matters for cost.”** Output tokens cost several times more than input, so trimming the output — structured format plus a length budget — is the bigger lever.

> 💡 **Info**
>
> ⚠️ Anti-patternTwo habits that quietly undo your work. **Shipping an optimization without measuring quality** — a cheaper prompt that answers worse costs you more than it saves; always A/B on the eval set. And **putting variable content early in the prompt** — caching only fires on an identical leading prefix, so anything dynamic up front cache-misses every call and you pay full price for the whole thing.

---

## 🎯 Concept 1: Why Bloat Costs Twice

### Why Bloat Costs Twice

Every token costs money on input and slows the model’s attention. And a system prompt runs on every request, so an 800-token system prompt at 10,000 requests a day is millions of wasted tokens daily - most prompts are 40-60 percent bloat. Trimming it is the highest-ROI cut in AI engineering.

Start with why this is the highest-leverage work you can do. Every token in a prompt costs money **twice**: once as **input** you are billed for, and once in the model’s **attention** computation (more tokens, slower processing). And the crucial multiplier is that a **system prompt runs on every single request**. An eight-hundred-token system prompt that could be two hundred is not a one-time waste of six hundred tokens; it is six hundred wasted tokens on *every call* — millions a day at scale, before the user types a word. Most prompts are **40 to 60 percent bloat**: redundant instructions, polite filler, examples you do not need, and context the model already knows. That is why trimming the prompt beats hunting for a cheaper model or a discount: the model and the discount help one call, but a leaner prompt helps *every* call, forever. The block shows the daily waste, keyless (illustrative prices).

> 🧰 **Analogy**
>
> It is an **overweight-baggage fee you pay on every flight**. A one-time overpacking is annoying; but if you fly the same route every day with the same overweight bag, that fee lands over and over. The system prompt is that bag: it flies on every request. Shave a few kilos once (trim it once) and you stop paying the fee on every single trip from now on — which is why a small, one-time edit to a high-traffic prompt is the best-paying hour in AI engineering.

An 800-token system prompt runs on 10,000 requests a day. You trim it to 200 tokens. How often does that saving apply?

**📝 `01_bloat_costs_twice.py`** - *runnable*

In [ ]:
# Illustrative price (USD per 1M tokens): flash input $0.15. Every token costs money on INPUT,
# and a system prompt runs on EVERY request - so bloat multiplies across all of them.
IN_PRICE = 0.15
bloated, concise = 800, 200          # same-quality system prompts, in tokens
requests_per_day = 10000

waste_tok = (bloated - concise) * requests_per_day
waste_usd_day = waste_tok * IN_PRICE / 1_000_000
print("A system prompt runs on every request. Two versions, same output quality:")
print("  bloated system prompt: {} tokens".format(bloated))
print("  concise system prompt: {} tokens  ({}x smaller)".format(concise, bloated // concise))
print()
print("At {:,} requests/day the {}-token difference is paid EVERY time:".format(requests_per_day, bloated - concise))
print("  wasted input tokens/day: {:,}".format(waste_tok))
print("  wasted cost/day (flash): ${:.2f}   -> ${:.2f}/month".format(waste_usd_day, waste_usd_day * 30))
print()
print("Bloat costs twice: once as input you pay for, once in slower attention. Most prompts are 40-60% bloat.")
print("Trimming the system prompt is the highest-ROI cut in AI engineering - it applies to every call, forever.")

# Output:
# A system prompt runs on every request. Two versions, same output quality:
#   bloated system prompt: 800 tokens
#   concise system prompt: 200 tokens  (4x smaller)
#
# At 10,000 requests/day the 600-token difference is paid EVERY time:
#   wasted input tokens/day: 6,000,000
#   wasted cost/day (flash): $0.90   -> $27.00/month
#
# Bloat costs twice: once as input you pay for, once in slower attention. Most prompts are 40-60% bloat.
# Trimming the system prompt is the highest-ROI cut in AI engineering - it applies to every call, forever.

- The bloated and concise system prompts produce the **same quality**, but one is four times the tokens of the other.
- The difference is paid on **every request**: at ten thousand a day, the six-hundred-token gap is millions of wasted input tokens daily.
- That is real money every day (and every month) for tokens that add nothing — and it costs **twice** (billing plus slower attention).
- The lesson: trimming a high-traffic prompt is the highest-ROI cut, because it applies to every call, forever.

#### 💡 What just happened

⚡ What just happened? Every token costs twice (input billing + attention), and a system prompt runs on every request - so an oversized system prompt wastes millions of tokens a day, and most prompts are 40-60 percent bloat. Tradeoff / the framing: trimming the prompt beats a cheaper model or a discount because it helps every call. Next: where the bloat actually hides.

- A requests-per-day slider; the 800-vs-200-token system prompt multiplies into a growing wasted-tokens bar
- The daily and monthly wasted cost climbs as you turn up the traffic

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Find the Waste: Analyze by Component

### Find the Waste: Analyze by Component

You cannot optimize what you have not measured. Break the prompt into its components - system prompt, few-shot examples, retrieved context, history, query - and count the tokens in each. The biggest and most redundant is the first target; the user query is usually a rounding error.

Before you cut anything, **measure**. A prompt is not one blob; it is a stack of components, each contributing tokens: the **system prompt**, the **few-shot examples**, the **retrieved context** (from RAG), the **conversation history**, and the **user query**. Count the tokens in each and the picture is almost always the same: the retrieved context and the system prompt dominate, the history is chunky, and the user’s actual query — the thing you were worried about — is a **rounding error**. The rule is to target the *biggest, most structurally redundant* component first, because that is where the tokens (and the money) are. A prompt analyzer that reports tokens by component turns “this prompt feels long” into “the retrieved context is two-thirds of it, start there.” The block analyzes a prompt by component, keyless.

> 📋 **Analogy**
>
> It is **an itemized packing list before you cut weight**. If your bag is overweight, you do not randomly pull things out — you lay everything on the bed and weigh each pile: the shoes, the books, the toiletries, the one pair of socks. Then you see that the books are eighty percent of the weight and the socks are nothing, so you know exactly what to cut. Analyzing a prompt by component is that weigh-in: it tells you the retrieved context is the heavy pile and the query is the socks.

You measure a RAG prompt by component. Which is almost always the biggest, and which is a rounding error?

**📝 `02_analyze_by_component.py`** - *runnable*

In [ ]:
# You cannot optimize what you have not measured. Break the prompt into components and count tokens.
components = {  # (illustrative token counts)
    "system prompt":     480,
    "few-shot examples": 240,
    "retrieved context": 3000,
    "conversation history": 800,
    "user query":        50,
}
total = sum(components.values())
print("Prompt anatomy ({} tokens total):".format(total))
for name, tok in sorted(components.items(), key=lambda kv: -kv[1]):
    bar = "#" * round(tok / total * 40)
    print("  {:<20} {:>5} tok  {:>4.0f}%  {}".format(name, tok, tok / total * 100, bar))
print()
biggest = max(components, key=components.get)
print("Biggest component: {} at {:.0f}% - and the most structurally redundant, so it is the FIRST target.".format(biggest, components[biggest] / total * 100))
print("The user query is {:.0f}% of the prompt - a rounding error. The tokens (and the money) are in the context and the system prompt.".format(components["user query"] / total * 100))

# Output:
# Prompt anatomy (4570 tokens total):
#   retrieved context     3000 tok    66%  ##########################
#   conversation history   800 tok    18%  #######
#   system prompt          480 tok    11%  ####
#   few-shot examples      240 tok     5%  ##
#   user query              50 tok     1%  
#
# Biggest component: retrieved context at 66% - and the most structurally redundant, so it is the FIRST target.
# The user query is 1% of the prompt - a rounding error. The tokens (and the money) are in the context and the system prompt.

- The prompt is broken into components and each is counted — the **retrieved context** is about two-thirds of the tokens.
- The **user query** is roughly one percent — a rounding error, so trimming it saves almost nothing.
- The biggest, most redundant component (the context) is the **first target**; that is where the money is.
- You get each count from a tokenizer — `tiktoken` offline, or the provider’s token-count endpoint — run on each component string; the counts here are illustrative, pre-measured values.
- The lesson: measure by component before you cut, so you spend your effort where the tokens actually are.

#### 💡 What just happened

⚡ What just happened? Analyze the prompt by component (system / few-shot / context / history / query) and count the tokens in each; the retrieved context and system prompt dominate while the query is a rounding error. Tradeoff: measuring takes a token count per component, in exchange for cutting where it actually pays. Next: the two manual trims that go furthest.

- A stacked bar of a prompt by component (system / few-shot / context / history / query)
- The biggest, most-redundant component lights up as the first target; the query is a sliver

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Trim the Input

### Trim the Input

The manual, highest-ROI cut: rewrite verbose instructions as concise single-purpose directives (about 30-50 percent off, and clearer), prune few-shot examples to the one to three that hold quality, and drop context the model already knows. Concise is cheaper and less ambiguous.

Now cut the input, starting with the two moves that go furthest. First, **concise instructions**: most system prompts are padded with redundancy and politeness (“please kindly ensure that you carefully…”). Rewriting to a **concise, single-purpose directive** — the same meaning with the filler removed — shaves roughly **thirty to fifty percent** off input usage, and it often makes the model *better*, because concise prompts have less ambiguity to trip over. Second, **few-shot pruning**: more examples do not always mean better output. Examples are expensive (each is hundreds of tokens), and most tasks reach the same quality with **one to three** demonstrations instead of five or ten — so prune to the minimum that holds. Third, **drop what the model already knows**: do not re-explain common formats, well-known facts, or things in the model’s training. And **trim the conversation history** — the second-largest component you measured in Step 2 — by keeping only the last few turns (a sliding window), replacing older turns with a short running summary, and dropping stale turns entirely; the deeper memory-management strategies are in Lesson 11.4. The block trims a verbose prompt and prunes the examples, keyless.

> ✍️ **Analogy**
>
> It is **editing a rambling email down to two lines**. The first draft says “I just wanted to quickly reach out and kindly ask whether it might possibly be feasible for you to perhaps consider…” The edited version says “Can you do X by Friday?” Same request, a fraction of the words, and *clearer* — the reader knows exactly what you want. A concise prompt is that edited email: you strip the hedging and filler, keep the one thing you are actually asking, and both the cost and the clarity improve.

You rewrite a verbose system prompt concisely and cut the few-shot from three examples to one. What happens to output quality?

**📝 `03_trim_input.py`** - *runnable*

In [ ]:
# The manual, highest-ROI cut: concise instructions + few-shot pruning. (illustrative token counts)
verbose = 120   # "Please kindly ensure that you carefully..." with redundancy + filler
concise = 55    # a single-purpose directive, same meaning
print("Trim 1 - concise instructions:")
print("  verbose instruction: {} tokens".format(verbose))
print("  concise instruction: {} tokens   ({:.0f}% smaller, same meaning, LESS ambiguous)".format(concise, (1 - concise / verbose) * 100))
print()
# Few-shot pruning: more examples do not always mean better output.
per_example = 200
before, after = 3, 1
print("Trim 2 - few-shot pruning ({} tokens/example):".format(per_example))
print("  {} examples: {} tokens".format(before, before * per_example))
print("  {} example:  {} tokens   ({:.0f}% smaller; 1-3 examples usually match 5+ on quality)".format(after, after * per_example, (1 - after / before) * 100))
print()
input_before = verbose + before * per_example
input_after = concise + after * per_example
print("Input tokens: {} -> {}  = {:.0f}% off the instruction + examples, quality held.".format(input_before, input_after, (1 - input_after / input_before) * 100))
print("Also drop context the model already knows (common formats, well-known facts) - do not re-explain it.")

# Output:
# Trim 1 - concise instructions:
#   verbose instruction: 120 tokens
#   concise instruction: 55 tokens   (54% smaller, same meaning, LESS ambiguous)
#
# Trim 2 - few-shot pruning (200 tokens/example):
#   3 examples: 600 tokens
#   1 example:  200 tokens   (67% smaller; 1-3 examples usually match 5+ on quality)
#
# Input tokens: 720 -> 255  = 65% off the instruction + examples, quality held.
# Also drop context the model already knows (common formats, well-known facts) - do not re-explain it.

- The **verbose instruction** drops to a concise one at about half the tokens — same meaning, less ambiguity.
- **Few-shot pruning** from three examples to one removes most of the example tokens; one to three usually match five-plus on quality.
- Together the instruction plus examples fall by roughly two-thirds of the input, with quality held.
- And **drop what the model knows** — do not spend tokens re-explaining common formats or well-known facts.
- **Trim the conversation history** (the second-largest component) with a sliding window or a rolling summary; the deeper memory strategies are in Lesson 11.4.

#### 💡 What just happened

⚡ What just happened? Trim the input with concise single-purpose instructions (~30-50 percent off, and clearer), few-shot pruning to 1-3 examples, and dropping context the model already knows. Tradeoff: you invest a rewrite once, in exchange for a leaner, clearer prompt on every call; verbosity buys ambiguity, not accuracy. Next: the output, which is the bigger lever.

- A verbose prompt with filler struck through down to a concise version, token counter falling
- A few-shot rack dropping from three examples to one, quality held

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Cut the Output

### Cut the Output

Output is the bigger lever, because output tokens cost several times more than input. Request structured output (about 30-50 percent fewer output tokens, since the model stops padding with prose), budget the response - ‘answer in 3 short bullets’ - and set max_tokens. Concise input plus controlled output stacks to 50-80 percent total savings.

Here is the move most people miss: **the output is the bigger lever**. Because output tokens are priced **three to six times** higher than input tokens, trimming the output length saves several times more per token than trimming the same amount of input. Three techniques. First, **structured output**: ask for JSON, a fixed schema, or bullet points instead of free-form prose, and the output token count drops by roughly **thirty to fifty percent**, because the model stops padding the answer with connective prose. Second, **budget the response**: instruct brevity (“answer in three short bullets”), and set a **`max_tokens`** cap so a runaway generation cannot blow the budget. Third, on a **reasoning model** the biggest output cost of all is the hidden chain-of-thought — recall from 13.1 that reasoning tokens are billed as output you never see, so a small visible answer can ride on several times as many hidden ones — so for a simple task, reach for a non-reasoning model or a lower reasoning effort and you cut the largest output component there is. A concise input *plus* a controlled output stacks to **fifty to eighty percent** total savings on a call. The block prices prose against structured output, keyless.

> 📝 **Analogy**
>
> It is **asking for bullet points, not an essay**. Ask a wordy colleague “how did the launch go?” and you get three rambling paragraphs; ask “give me the top three outcomes in one line each” and you get exactly what you need in a tenth of the words. The model is that colleague: given no shape, it pads with prose; given a structure and a length (“three short bullets”), it delivers the substance and skips the filler. And because each of those output words is the expensive kind, cutting them saves the most.

You cut 150 tokens from the output and 150 tokens from the input of the same call. Which saves more money?

**📝 `04_cut_output.py`** - *runnable*

In [ ]:
# Output is the BIGGER lever: output tokens cost more per token, so trimming output beats trimming input.
IN_PRICE, OUT_PRICE = 0.15, 0.60     # flash illustrative; output is 4x input per token
prose_out, structured_out = 300, 150 # same information, free-form prose vs a structured answer

print("Same answer, two formats:")
print("  free-form prose: {} output tokens".format(prose_out))
print("  structured (bullets/JSON): {} output tokens   ({:.0f}% fewer - the model stops padding with prose)".format(structured_out, (1 - structured_out / prose_out) * 100))
print()
saved_out = (prose_out - structured_out) * OUT_PRICE / 1_000_000
saved_in_same = (prose_out - structured_out) * IN_PRICE / 1_000_000
print("Trimming {} OUTPUT tokens saves ${:.6f}".format(prose_out - structured_out, saved_out))
print("The SAME {}-token cut on INPUT would save only ${:.6f} - {:.0f}x less, because output is priced {:.0f}x higher.".format(prose_out - structured_out, saved_in_same, saved_out / saved_in_same, OUT_PRICE / IN_PRICE))
print()
print("So: request structured output, budget the response ('answer in 3 short bullets'), and set max_tokens.")
print("Concise input + controlled output stacks to ~50-80% total savings.")

# Output:
# Same answer, two formats:
#   free-form prose: 300 output tokens
#   structured (bullets/JSON): 150 output tokens   (50% fewer - the model stops padding with prose)
#
# Trimming 150 OUTPUT tokens saves $0.000090
# The SAME 150-token cut on INPUT would save only $0.000023 - 4x less, because output is priced 4x higher.
#
# So: request structured output, budget the response ('answer in 3 short bullets'), and set max_tokens.
# Concise input + controlled output stacks to ~50-80% total savings.

- **Structured output** (bullets/JSON) cuts the output token count by about half versus free-form prose — the model stops padding.
- Because output is priced about four times higher than input, trimming those output tokens saves **four times more** than the same cut on input.
- So the levers are: request **structured output**, **budget the response length**, and set **max_tokens** as a hard cap.
- On a **reasoning model** the hidden chain-of-thought is the biggest output cost (recall 13.1), so use a non-reasoning model or a lower reasoning effort for a simple task.
- A concise input plus a controlled output stacks to roughly fifty to eighty percent total savings on a call.

#### 💡 What just happened

⚡ What just happened? Output is the bigger cost lever because output tokens cost 3-6x input; cut it with structured output (~30-50 percent fewer tokens), a length budget, a max_tokens cap, and - on reasoning models - a lighter model or lower reasoning effort, since the hidden chain-of-thought is billed as output. Tradeoff: a tighter output format constrains the answer’s shape, in exchange for the largest per-token saving. Next: compressing the parts you cannot hand-trim.

- A prose answer vs a structured/bulleted answer; the output bar shrinks about half
- Because output is priced 4x, its dollar bar shrinks even more than the input equivalent

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Automated Compression: LLMLingua

### Automated Compression: LLMLingua

When a prompt is too long to hand-trim - a big retrieved context, a long history - a compressor scores each token’s informativeness with a small model and drops the low-value ones, with a budget controller keeping different ratios per component. LLMLingua reached about 9x with under 2 percent quality loss.

Manual trimming works for instructions and examples, but some prompts are too long to hand-edit — a two-thousand-token retrieved context, a long conversation history. For those, **automated prompt compression** does the trimming for you. The idea (LLMLingua, from Microsoft): a **small, cheap model scores each token’s informativeness** and drops the low-value ones below a threshold, keeping the tokens that actually carry the meaning. A **budget controller** applies *different* compression ratios to different parts — keeping more of the instruction, aggressively compressing the demonstrations. The results are striking: LLMLingua compressed a prompt from **847 to 94 tokens (about nine times), dropping input cost by the same factor, with under two percent quality loss** — and up to twenty times on some workloads, while also cutting latency. LLMLingua-2 reframes this as a simple keep-or-drop tag on each token, learned by copying the choices of a larger model. Compression is **lossy**, so you always validate it (Step 7). The block models the compressor, keyless.

> 📦 **Analogy**
>
> It is a **zip file that keeps the meaning**. A good compressor does not throw away random bytes; it finds the redundancy and squeezes it out so the file is a fraction of the size but unzips to the same content. LLMLingua is that for prompts: a small model reads the token stream, spots the low-information filler (the redundant phrasing, the padding in a long context), and drops it, keeping the tokens that carry the signal — a nine-to-one squeeze that the big model can still act on almost as well.

An automated compressor like LLMLingua reduces a prompt from 847 to 94 tokens. How does it decide what to keep?

**📝 `05_compression.py`** - *runnable*

In [ ]:
# When a prompt is too long to hand-trim, a compressor (LLMLingua) scores each token's
# informativeness with a small model and drops the low-value ones. A BUDGET CONTROLLER keeps
# different ratios per component (instructions high, demonstrations low).
IN_PRICE = 0.15
parts = {  # (tokens, keep_ratio) - the budget controller keeps more of the instruction, less of demos
    "instruction":    (240, 0.25),
    "demonstrations": (480, 0.04),
    "question":       (127, 0.12),
}
orig = sum(t for t, _ in parts.values())
kept = sum(round(t * r) for t, r in parts.values())
print("LLMLingua-style compression (keep the informative tokens, drop the filler):")
for name, (t, r) in parts.items():
    print("  {:<15} {:>4} tok -> keep {:>2.0f}% -> {:>3} tok".format(name, t, r * 100, round(t * r)))
print()
ratio = orig / kept
print("Total: {} -> {} tokens = {:.1f}x compression".format(orig, kept, ratio))
print("  input cost drops the same {:.1f}x: ${:.6f} -> ${:.6f}".format(ratio, orig * IN_PRICE / 1_000_000, kept * IN_PRICE / 1_000_000))
print("  measured quality loss: under 2% (illustrative) - compression is lossy, so you VALIDATE it (Step 7).")
print("Use it for the parts you cannot hand-trim: a big retrieved context, a long history.")

# Output:
# LLMLingua-style compression (keep the informative tokens, drop the filler):
#   instruction      240 tok -> keep 25% ->  60 tok
#   demonstrations   480 tok -> keep  4% ->  19 tok
#   question         127 tok -> keep 12% ->  15 tok
#
# Total: 847 -> 94 tokens = 9.0x compression
#   input cost drops the same 9.0x: $0.000127 -> $0.000014
#   measured quality loss: under 2% (illustrative) - compression is lossy, so you VALIDATE it (Step 7).
# Use it for the parts you cannot hand-trim: a big retrieved context, a long history.

- A **budget controller** keeps different ratios per component — more of the instruction, very little of the demonstrations.
- The prompt collapses from **847 to 94 tokens** (about nine times), and the input cost drops by the same factor.
- The measured **quality loss is under two percent** — the compressor kept the informative tokens and dropped the filler.
- Compression is **lossy**, so you use it for the un-hand-trimmable parts (a big context, a long history) and always validate it.

#### 💡 What just happened

⚡ What just happened? Automated compression (LLMLingua) scores each token’s informativeness with a small model and drops the low-value ones, with a budget controller per component - about 9x reduction at under 2 percent quality loss. Tradeoff: compression is lossy and adds a small pre-processing step, in exchange for cutting contexts you could never hand-trim. Next: structuring the prompt so the repeated part is cached.

- A prompt of tokens each with an informativeness score; drag a threshold
- Low-value tokens drop, 847 collapsing toward 94 with a quality meter barely moving

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Structure for the Cache

### Structure for the Cache

Prompt caching charges the cached prefix at a fraction of the price - but only if it is identical and comes first. So structure the prompt as a stable prefix (system prompt + few-shot) followed by the dynamic suffix (context + query). Put anything variable early and you cache-miss every call.

Optimizing a prompt’s *shape* is as valuable as trimming its length. Recall from 12.4 that prompt caching charges a cached prefix at roughly **a tenth** of the input price — but it only fires when the cached prefix is **identical and comes first**. That gives you a structural rule: split the prompt into a **stable prefix** (the system prompt and the few-shot examples, which never change from call to call) and a **dynamic suffix** (the retrieved context and the user query, which change every time). Put the stable block **first** and the expensive, repeated prefix is cached at a fraction of the cost on every call. Put anything variable early — a timestamp, the user’s name, today’s context — and the leading prefix is no longer identical, so you **cache-miss every request** and pay full price for the whole thing. The block prices the two layouts, keyless.

> 🥪 **Analogy**
>
> It is **shelving the staples up front and the fresh stuff at the back**. A good kitchen keeps the things it uses on every dish — salt, oil, the standing mise en place — in the same fixed spot it never rearranges (the stable, cached prefix), and the day’s specials on a separate board that changes (the dynamic suffix). If you reshuffled the whole pantry every service because you tucked today’s specials in among the staples, you would waste time re-finding everything each time. Prompt structure is that discipline: keep the unchanging block fixed and first, so it is always ‘already there’ (cached).

You put the user’s query and a timestamp at the START of your prompt, before the system prompt. What happens to caching?

**📝 `06_cache_structure.py`** - *runnable*

In [ ]:
# Prompt caching (12.4) charges the cached prefix at ~10% of input price - but ONLY if it is
# IDENTICAL and comes FIRST. So structure the prompt: stable prefix, then dynamic suffix.
IN_PRICE = 0.15
CACHED_PRICE = IN_PRICE * 0.10       # cached input ~10% of input
stable = 3500     # system prompt + few-shot examples - never change
dynamic = 100     # the user query - changes every call

# Layout A: stable prefix FIRST (cacheable) + dynamic suffix (fresh)
hit = (stable * CACHED_PRICE + dynamic * IN_PRICE) / 1_000_000
# Layout B: something variable early -> the prefix is not identical -> CACHE MISS (all fresh)
miss = (stable + dynamic) * IN_PRICE / 1_000_000

print("A prompt = a {}-token STABLE prefix (system + few-shot) + a {}-token DYNAMIC suffix (query):".format(stable, dynamic))
print("  stable-prefix-FIRST (cache HIT):  ${:.6f}   (the {}-token prefix is cached at 10% of input)".format(hit, stable))
print("  variable-content-first (cache MISS): ${:.6f}   (prefix not identical -> nothing cached, all fresh)".format(miss))
print()
print("  structuring the prompt saves ${:.6f} PER REQUEST - about {:.1f}x cheaper on the repeated prefix.".format(miss - hit, miss / hit))
print("Put anything variable early and you cache-miss every call. Optimizing the prompt's SHAPE matters as much as its length.")

# Output:
# A prompt = a 3500-token STABLE prefix (system + few-shot) + a 100-token DYNAMIC suffix (query):
#   stable-prefix-FIRST (cache HIT):  $0.000068   (the 3500-token prefix is cached at 10% of input)
#   variable-content-first (cache MISS): $0.000540   (prefix not identical -> nothing cached, all fresh)
#
#   structuring the prompt saves $0.000472 PER REQUEST - about 8.0x cheaper on the repeated prefix.
# Put anything variable early and you cache-miss every call. Optimizing the prompt's SHAPE matters as much as its length.

- With the **stable prefix first**, the big system-plus-few-shot block is **cached at a tenth** of the input price — a cache hit.
- With **variable content first**, the leading prefix is not identical, so **nothing caches** and you pay full price for the whole prompt — a cache miss.
- Structuring the prompt saves real money **on every request** — several times cheaper on the repeated prefix.
- The rule: stable prefix (system + few-shot) first, dynamic suffix (context + query) last; never put anything variable up front.

#### 💡 What just happened

⚡ What just happened? Caching only fires on an identical leading prefix, so structure the prompt as a stable prefix (system + few-shot) first and a dynamic suffix (context + query) last - variable-first cache-misses every call. Tradeoff: you commit to a prompt layout, in exchange for a cached, fraction-of-price prefix on every request. Optimizing the prompt’s shape matters as much as its length. Next: proving the cut was worth it.

- Drag the stable prefix (system + few-shot) before or after the dynamic part (context + query)
- Prefix-first lights a green cache hit at a fraction of the cost; interleaved lights a rose miss

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Measure the Tradeoff

### Measure the Tradeoff

Never ship a cut on faith. A/B the optimized prompt against the original on an eval set and keep it only if quality holds within tolerance; reject a cut that drops below the floor. Enforce a token budget in the template so bloat cannot creep back. Cost down, quality held - measured, not assumed.

Every optimization so far *risks quality*, so the final discipline is to never ship a cut on faith. **A/B the optimized prompt against the original on a fixed eval set** — the same golden signal / eval set you met in 12.8 and build in 14.4 — and keep the cheaper prompt **only if its quality holds** within tolerance. A prompt that is half the cost but scores meaningfully worse is a *net loss*, not a win. So a modest cut that holds quality ships; an aggressive compression that drops below the quality floor is **rejected**, even though it saves more. And to stop bloat from creeping back over time, enforce a **token budget in the template**: assert that the assembled prompt is under a cap, and fail the build if a well-meaning edit pushes it over. The whole lesson in one line: **cost down, quality held — measured, not assumed**. The block runs the A/B and the guardrail, keyless.

> 😋 **Analogy**
>
> It is a **taste test before you change the recipe**. A restaurant wanting to cut costs does not just swap in the cheaper cheese and hope — they cook both versions and taste them side by side. If the cheaper one tastes the same, it ships and they pocket the saving; if it tastes worse, they keep the original, because a dish nobody orders saves nothing. A/B-ing a prompt is that taste test: the cheaper prompt only earns its place if the eval score survives the cut.

An optimization makes a prompt 56 percent cheaper but its eval score drops from 8.6 to 7.2, below your 8.0 floor. Do you ship it?

**📝 `07_measure_tradeoff.py`** - *runnable*

In [ ]:
# Never ship a cut on faith. A/B the optimized prompt against the original on an EVAL SET,
# and keep it ONLY if quality holds. Plus a token-budget guardrail so bloat cannot creep back.
QUALITY_FLOOR = 8.0
original = {"tokens": 800, "quality": 8.6}
candidates = {
    "concise + structured":     {"tokens": 350, "quality": 8.5},   # big cut, quality holds
    "aggressive compression":   {"tokens": 180, "quality": 7.2},   # too lossy
}
print("A/B each optimization vs the original (original: {} tokens, quality {}/10):".format(original["tokens"], original["quality"]))
for name, c in candidates.items():
    saved = (1 - c["tokens"] / original["tokens"]) * 100
    keep = c["quality"] >= QUALITY_FLOOR and c["quality"] >= original["quality"] - 0.3
    verdict = "KEEP" if keep else "REJECT (quality below the {} floor / tolerance)".format(QUALITY_FLOOR)
    print("  {:<24} {} tok ({:.0f}% off), quality {} -> {}".format(name, c["tokens"], saved, c["quality"], verdict))
print()
# Token-budget guardrail in the template: reject an over-budget assembly.
BUDGET = 500
for assembled in [420, 620]:
    ok = assembled <= BUDGET
    print("  token-budget guardrail: assembled prompt {} tokens vs {} budget -> {}".format(assembled, BUDGET, "OK" if ok else "REJECT (over budget)"))
print()
print("Cost down, quality held - MEASURED, not assumed. Keep the win only when the eval score survives the cut.")

# Output:
# A/B each optimization vs the original (original: 800 tokens, quality 8.6/10):
#   concise + structured     350 tok (56% off), quality 8.5 -> KEEP
#   aggressive compression   180 tok (78% off), quality 7.2 -> REJECT (quality below the 8.0 floor / tolerance)
#
#   token-budget guardrail: assembled prompt 420 tokens vs 500 budget -> OK
#   token-budget guardrail: assembled prompt 620 tokens vs 500 budget -> REJECT (over budget)
#
# Cost down, quality held - MEASURED, not assumed. Keep the win only when the eval score survives the cut.

**📝 `optimize-pipeline.py`** - *illustrative*

*Illustrative prompt-optimization pipeline (compress + structure + budget). Shown for reference only — it needs llmlingua + anthropic + an API key + an eval set, so it is not executed here.*

```python
# PROMPT OPTIMIZATION PIPELINE - an illustrative minimal example (compress + structure + budget).
from llmlingua import PromptCompressor
import anthropic, tiktoken

client = anthropic.Anthropic()
compressor = PromptCompressor()          # small-model token-informativeness scorer
enc = tiktoken.get_encoding("cl100k_base")            # offline tokenizer - the measure-first primitive from Step 2
def count_tokens(text): return len(enc.encode(text))  # count each component before you cut

STABLE_PREFIX = "You are a support agent. " + FEW_SHOT   # never changes -> cacheable, goes FIRST
TOKEN_BUDGET = 1500

def optimized_answer(retrieved_context, query):
    # 1) compress the big, un-hand-trimmable retrieved context (keep the informative tokens)
    ctx = compressor.compress_prompt(retrieved_context, rate=0.3)["compressed_prompt"]
    # 2) assemble stable-prefix-first (cache hit) + dynamic suffix; ask for STRUCTURED, short output
    prompt = STABLE_PREFIX + "\n\nContext:\n" + ctx + "\n\nQ: " + query + "\nAnswer in 3 short bullets."
    assert count_tokens(prompt) <= TOKEN_BUDGET, "prompt over token budget"   # guardrail
    resp = client.messages.create(model="claude-sonnet-4-6", max_tokens=256,  # cap the output
                                  messages=[{"role": "user", "content": prompt}])
    return resp.content[0].text
# Then A/B this against the original on your eval set; ship only if quality holds (Step 7).
# Output: (illustrative minimal example - needs llmlingua + anthropic + a key + an eval set; not run here.)
```

- Each optimization is **A/B-tested** against the original on an eval set: the concise-plus-structured cut holds quality and is **kept**.
- The **aggressive compression** saves more but drops below the quality floor, so it is **rejected** — cheaper-but-worse is a net loss.
- A **token-budget guardrail** in the template rejects an over-budget assembly, so bloat cannot creep back over time.
- The **illustrative pipeline** ties it together: compress the context, assemble stable-prefix-first, ask for short structured output, assert the budget, then A/B before shipping.

#### 💡 What just happened

⚡ What just happened? Never ship a cut on faith: A/B the optimized prompt on an eval set and keep it only if quality holds, reject it below the floor, and enforce a token budget so bloat cannot return. Tradeoff: measuring costs an eval run per candidate, in exchange for never trading real quality for a few tokens. That is the whole lesson: cost down, quality held, measured.

- An A/B panel: the optimized prompt’s cost bar drops while a quality gauge checks against a floor
- Quality held = keep (emerald); dropped below the floor = reject (rose)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Prompt optimization is a pipeline of levers, each one measured. It starts from the law that **bloat costs twice and runs on every request**, so trimming the prompt is the highest-ROI cut (Step 1). You **analyze by component** to find where the tokens hide — usually the retrieved context and the system prompt, not the query (Step 2). You **trim the input** with concise instructions and few-shot pruning (Step 3), and **cut the output** with structured formats and a length budget — the bigger lever, since output is priced several times higher (Step 4). For the contexts you cannot hand-trim, an **automated compressor** scores tokens by informativeness and squeezes them (Step 5). You **structure the prompt** stable-prefix-first so the repeated part is cached (Step 6). And you **measure every cut** against an eval set, keeping the win only when quality holds, with a token budget to stop bloat creeping back (Step 7). Ask two questions of any prompt optimization: did it **cut tokens meaningfully**, and did **quality hold on the eval set**?

| Lever | Typical saving | The risk to watch |
|---|---|---|
| Concise instructions | ~30-50 percent input | losing an essential constraint |
| Few-shot pruning | example tokens | dropping a needed demonstration |
| Trim conversation history | older turns | losing needed context |
| Structured output | ~30-50 percent output | over-constraining the answer |
| Length budget / max_tokens | output tokens | truncating a real answer |
| Automated compression | up to ~9x on context | lossy - must be validated |
| Cache-aligned structure | prefix at ~10 percent | any variable content up front |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now cut a prompt’s tokens without cutting its quality. The **eval set** your A/B test depends on — error analysis and eval-driven development — is built in Lesson 14.4, and the **caching mechanics** your cache-aligned structure targets are in Lesson 12.4. The **cost model** behind every token is Lesson 13.1, and the next cost levers — model routing, distillation, and serving economics — come later in Module 13. The primary references are the [Microsoft LLMLingua](https://www.microsoft.com/en-us/research/blog/llmlingua-innovating-llm-efficiency-with-prompt-compression/) work and its [source](https://github.com/microsoft/LLMLingua), [Morph on prompt compression](https://www.morphllm.com/prompt-compression), [Anthropic prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching), and [tiktoken](https://github.com/openai/tiktoken) for counting tokens per component.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **PromptOptimization**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-13_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-13.2.html` - regenerate this notebook whenever the source HTML is updated.*
